# Gradient Descent Audio Denoising
Ini adalah notebook yang memuat implementasi *from scratch* untuk reduksi noise pada sinyal audio menggunakan metode Gradient Descent.

In [ ]:
import os
import numpy as np
import time
from scipy.io import wavfile
import matplotlib.pyplot as plt
from IPython.display import Audio, display


In [ ]:
class AudioProcessor:
    @staticmethod
    def load_audio(filepath):
        try:
            sample_rate, data = wavfile.read(filepath)
            if len(data.shape) > 1 and data.shape[1] > 1:
                data = data.mean(axis=1)
            if data.dtype == np.int16:
                data_normalized = data.astype(np.float64) / 32768.0
            elif data.dtype == np.int32:
                data_normalized = data.astype(np.float64) / 2147483648.0
            else:
                data_normalized = data.astype(np.float64)
                max_val = np.max(np.abs(data_normalized))
                if max_val > 0:
                    data_normalized = data_normalized / max_val
            return sample_rate, data_normalized
        except Exception as e:
            print(f"Error saat memuat file audio: {e}")
            return None, None

    @staticmethod
    def save_audio(filepath, sample_rate, data_normalized):
        try:
            data_clipped = np.clip(data_normalized, -1.0, 1.0)
            data_int16 = np.int16(data_clipped * 32767.0)
            wavfile.write(filepath, sample_rate, data_int16)
        except Exception as e:
            print(f"Error saat menyimpan file audio: {e}")


In [ ]:
class GradientDescentSmoother:
    def __init__(self, learning_rate=0.1, max_epochs=100, lambda_reg=5.0, tol=1e-5):
        self.learning_rate = learning_rate
        self.max_epochs = max_epochs
        self.lambda_reg = lambda_reg
        self.tol = tol
        self.loss_history = []
        
    def _compute_gradient(self, X, Y):
        grad = X - Y
        grad[1:-1] += self.lambda_reg * (2 * X[1:-1] - X[:-2] - X[2:])
        grad[0] += self.lambda_reg * (X[0] - X[1])
        grad[-1] += self.lambda_reg * (X[-1] - X[-2])
        return grad

    def _compute_cost(self, X, Y):
        data_fidelity = 0.5 * np.sum((X - Y)**2)
        regularization = 0.5 * self.lambda_reg * np.sum((X[1:] - X[:-1])**2)
        return data_fidelity + regularization

    def denoise(self, noisy_signal):
        Y = noisy_signal
        X = np.copy(Y)
        self.loss_history = []
        print(f"Memulai iterasi Gradient Descent (maks {self.max_epochs} epochs)...")
        start_time = time.time()
        for epoch in range(self.max_epochs):
            grad = self.compute_gradient(X, Y)
            X = X - (self.learning_rate * grad)
            cost = self.compute_cost(X, Y)
            self.loss_history.append(cost)
            if epoch % 10 == 0 or epoch == self.max_epochs - 1:
                print(f"Epoch {epoch:4d} | Cost: {cost:.4f}")
            if epoch > 0 and abs(self.loss_history[-2] - cost) < self.tol:
                print(f"Konvergensi tercapai pada epoch {epoch}.")
                break
        elapsed = time.time() - start_time
        print(f"Proses optimasi selesai dalam {elapsed:.2f} detik.")
        return X
    
    compute_gradient = _compute_gradient
    compute_cost = _compute_cost


In [ ]:
class Visualizer:
    @staticmethod
    def plot_comparison(original_signal, cleaned_signal, sample_rate, filepath=None):
        max_samples = min(len(original_signal), sample_rate * 1) 
        y_orig = original_signal[:max_samples]
        y_clean = cleaned_signal[:max_samples]
        time_axis = np.linspace(0, max_samples / sample_rate, num=max_samples)
        
        plt.figure(figsize=(12, 8))
        plt.subplot(2, 1, 1)
        plt.title('Before: Sinyal Audio Asli (dengan Noise)')
        plt.plot(time_axis, y_orig, color='red', alpha=0.7)
        plt.xlabel('Waktu (detik)')
        plt.ylabel('Amplitudo')
        plt.grid(True)
        
        plt.subplot(2, 1, 2)
        plt.title('After: Sinyal Audio Bersih (Pasca Gradient Descent)')
        plt.plot(time_axis, y_clean, color='blue', alpha=0.7)
        plt.xlabel('Waktu (detik)')
        plt.ylabel('Amplitudo')
        plt.grid(True)
        plt.tight_layout()
        if filepath:
            plt.savefig(filepath, dpi=300)
            print(f"Visualisasi perbandingan berhasil disimpan di: {filepath}")
        plt.show()

    @staticmethod
    def plot_loss(loss_history, filepath=None):
        plt.figure(figsize=(8, 5))
        plt.title('Kurva Penurunan Cost (Gradient Descent Convergence)')
        plt.plot(loss_history, color='green', linewidth=2)
        plt.xlabel('Epoch (Iterasi)')
        plt.ylabel('Cost Value (Tikhonov Loss)')
        plt.grid(True)
        plt.tight_layout()
        if filepath:
            plt.savefig(filepath, dpi=300)
            print(f"Visualisasi kurva loss berhasil disimpan di: {filepath}")
        plt.show()


In [ ]:
# Konfigurasi parameter
input_path = "../data/audio.wav"
output_audio_path = "../data/cleaned_sample.wav"
epochs = 500
lr = 0.01
lam = 20.0

print(f"--- Gradient Descent Audio Denoising ---")
print(f"Membaca file: {input_path}")

sample_rate, clean_signal = AudioProcessor.load_audio(input_path)

if clean_signal is not None:
    print(f"Audio termuat. Sample Rate: {sample_rate} Hz, Durasi: {len(clean_signal)/sample_rate:.2f} detik.")

    print("Menerapkan noise pada audio...")
    noise_strength = 0.02
    noisy_signal = clean_signal + noise_strength * np.random.randn(len(clean_signal))
    noisy_signal = np.clip(noisy_signal, -1, 1)

    smoother = GradientDescentSmoother(learning_rate=lr, max_epochs=epochs, lambda_reg=lam, tol=1e-6)
    cleaned_signal = smoother.denoise(noisy_signal)

    print(f"Menyimpan audio bersih ke: {output_audio_path}")
    AudioProcessor.save_audio(output_audio_path, sample_rate, cleaned_signal)

    print("Mengekspor gambar perbandingan visual...")
    Visualizer.plot_comparison(noisy_signal, cleaned_signal, sample_rate)
    Visualizer.plot_loss(smoother.loss_history)
    
    print("\nAUDIO KOTOR (DENGAN NOISE):")
    display(Audio(noisy_signal, rate=sample_rate))
    
    print("\nAUDIO BERSIH (DENOISED):")
    display(Audio(cleaned_signal, rate=sample_rate))
    
    print("Proses selesai dengan sukses!")
